In [ ]:
from dotenv import load_dotenv
from pprint import pprint
import requests
import os
import csv
from datetime import datetime

# Load the environment variables
load_dotenv()



companies = ['airbnb', 'stripe', 'gitlab', 'robinhood', 'coinbase', 'databricks', 'figma']

In [33]:
# Test if a company uses Greenhouse
def check_greenhouse(company):
    url = f"https://boards-api.greenhouse.io/v1/boards/{company}/jobs"
    try:
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except:
        return False

greenhouse_companies = [c for c in companies if check_greenhouse(c)]

print(greenhouse_companies)

['airbnb', 'stripe', 'gitlab', 'robinhood', 'coinbase', 'databricks', 'figma']


In [34]:
# Get job postings from Greenhouse given company name and write to CSV.
# CSV columns: company, absolute_url, title, content, fetched_date

def get_job_postings(company):
    url = f"https://boards-api.greenhouse.io/v1/boards/{company}/jobs"
    params = {
        'content': 'true'
    }
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()

    print(f"Status Code: {response.status_code}")
    data = response.json()
    jobs = data.get('jobs', [])
    
    # Get current date/time for fetched_date
    fetched_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # Filter to only include absolute_url, title, and content
    filtered_jobs = [
        {
            'company': company,
            'absolute_url': job.get('absolute_url'),
            'title': job.get('title'),
            'content': job.get('content'),
            'fetched_date': fetched_date
        }
        for job in jobs
    ]
    
    # Write to CSV
    csv_filename = 'job_postings.csv'
    file_exists = os.path.exists(csv_filename)
    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['company', 'absolute_url', 'title', 'content', 'fetched_date']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        # Write header if file is new
        if not file_exists:
            writer.writeheader()
        
        # Write all jobs
        writer.writerows(filtered_jobs)
    
    print(f"Wrote {len(filtered_jobs)} jobs to {csv_filename}")
    return len(filtered_jobs)

In [35]:
job_count = get_job_postings('airbnb')
print(f"Total jobs fetched: {job_count}")

Status Code: 200
Wrote 165 jobs to job_postings.csv
Total jobs fetched: 165
